In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, when, year, to_date, regexp_replace, explode, split, trim, upper, lit,
    last, expr, coalesce, try_to_date, initcap, avg as _avg, round as _round,
    min as _min, max as _max
)
from datetime import timedelta

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# 1) silver.tb_info_filmes
df_info = spark.read.table("bronze.tb_movies_info")
df_info = df_info.withColumn(
    "rn", expr("row_number() over (partition by id order by ingestion_datetime desc)")
).filter(col("rn") == 1).drop("rn", "ingestion_datetime")

# normalizacao antes da traducao
df_info = df_info.withColumn(
    "status_norm", upper(trim(regexp_replace(col("status"), r"[-_]+", " ")))
)
df_info = df_info.withColumn("status_filme",
    when(col("status_norm") == "RELEASED", "Lançado")
    .when(col("status_norm") == "POST PRODUCTION", "Pós-Produção")
    .when(col("status_norm") == "IN PRODUCTION", "Em Produção")
    .when(col("status_norm") == "PLANNED", "Planejado")
    .when(col("status_norm") == "RUMORED", "Rumores")
    .when(col("status_norm") == "CANCELED", "Cancelado")
    .otherwise("Não Informado")
).drop("status_norm")

# tratamento multi-formato de data
df_info = df_info.withColumn("data_lancamento",
    coalesce(
        try_to_date(col("release_date"), "yyyy-MM-dd"),
        try_to_date(col("release_date"), "MM-dd-yyyy"),
        try_to_date(col("release_date"), "dd/MM/yyyy"),
        try_to_date(col("release_date"), "yyyy/MM/dd"),
        try_to_date(col("release_date"), "dd-MM-yyyy")
    )
).withColumn("ano_lancamento", year("data_lancamento"))

df_info.selectExpr(
    "id as id_filme", "title as titulo", "original_title as titulo_original",
    "data_lancamento", "ano_lancamento", "runtime as duracao_minutos",
    "original_language as idioma_original", "status_filme",
    "overview as sinopse", "tagline as frase_divulgacao"
).write.format("delta").mode("overwrite") \
 .option("overwriteSchema", "true") \
 .saveAsTable("silver.tb_info_filmes")

# 7) silver.tb_cotacao_dolar
df_cotacao = spark.read.table("bronze.tb_cotacao_dolar") \
    .withColumn("dataHoraCotacao", to_date("dataHoraCotacao"))
df_cotacao_dia = df_cotacao.groupBy("dataHoraCotacao") \
    .agg(last("cotacaoCompra", ignorenulls=True).alias("cotacaoCompra"))
bounds = df_cotacao_dia.agg(
    _min("dataHoraCotacao").alias("dt_min"),
    _max("dataHoraCotacao").alias("dt_max")
).collect()[0]
datas, d = [], bounds["dt_min"]
while d <= bounds["dt_max"]:
    datas.append((d,)); d += timedelta(days=1)
calendario = spark.createDataFrame(datas, ["dataHoraCotacao"])
window_ff = Window.partitionBy(lit(1)).orderBy("dataHoraCotacao") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_cotacao_filled = calendario \
    .join(df_cotacao_dia, on="dataHoraCotacao", how="left") \
    .orderBy("dataHoraCotacao") \
    .withColumn("cotacaoCompra", last("cotacaoCompra", ignorenulls=True).over(window_ff))
df_cotacao_filled.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_cotacao_dolar")

# media do periodo
cot_media = df_cotacao_filled.agg(_avg("cotacaoCompra").alias("c")).collect()[0]["c"]

# 2) silver.tb_financeiro_filmes
df_fin = spark.read.table("bronze.tb_movies_financials")

def _limpar(campo):
    return expr(f"""
        try_cast(
            nullif(
                regexp_replace(
                    regexp_replace(upper(trim({campo})),
                        'UNKNOWN|NÃO INFORMADO|NAO INFORMADO|N/A|NULL|NONE', ''),
                    '[^0-9.]', ''),
            '')
        as decimal(18,2))
    """)

df_fin = df_fin \
    .withColumn("orcamento_usd", _limpar("budget")) \
    .withColumn("receita_usd", _limpar("revenue")) \
    .withColumn("orcamento_usd", when(col("orcamento_usd") > 0, col("orcamento_usd")).otherwise(None)) \
    .withColumn("receita_usd", when(col("receita_usd") > 0, col("receita_usd")).otherwise(None)) \
    .withColumn("orcamento_brl", _round(col("orcamento_usd") * lit(cot_media), 2)) \
    .withColumn("receita_brl", _round(col("receita_usd") * lit(cot_media), 2)) \
    .withColumn("lucro_usd", col("receita_usd") - col("orcamento_usd")) \
    .withColumn("lucro_brl", col("receita_brl") - col("orcamento_brl")) \
    .withColumn("margem_lucro_pct",
        when(col("receita_usd") > 0,
             _round((col("lucro_usd") / col("receita_usd")) * 100, 2)
        ).otherwise(None))

df_fin.selectExpr(
    "id as id_filme", "orcamento_usd", "receita_usd",
    "orcamento_brl", "receita_brl", "lucro_usd", "lucro_brl", "margem_lucro_pct"
).write.format("delta").mode("overwrite") \
 .option("overwriteSchema", "true") \
 .saveAsTable("silver.tb_financeiro_filmes")

# 3) silver.tb_metricas_engajamento
# formato estadunidense
df_met = spark.read.table("bronze.tb_movies_metrics")
df_met = df_met \
    .withColumn("popularidade", expr("try_cast(nullif(regexp_replace(popularity, '[^0-9.]', ''), '') as double)")) \
    .withColumn("nota_media_tmdb", expr("try_cast(nullif(regexp_replace(vote_average, '[^0-9.]', ''), '') as decimal(4,1))")) \
    .withColumn("qtd_votos_tmdb", expr("try_cast(nullif(regexp_replace(vote_count, '[^0-9]', ''), '') as int)")) \
    .withColumn("nota_media_imbd", expr("try_cast(nullif(regexp_replace(averageRating, '[^0-9.]', ''), '') as decimal(4,1))")) \
    .withColumn("qtd_votos_imbd", expr("try_cast(nullif(regexp_replace(numVotes, '[^0-9]', ''), '') as int)")) \
    .withColumn("popularidade", when(col("popularidade") >= 0, col("popularidade")).otherwise(None)) \
    .withColumn("nota_media_tmdb", when((col("nota_media_tmdb") >= 0) & (col("nota_media_tmdb") <= 10), col("nota_media_tmdb")).otherwise(None)) \
    .withColumn("nota_media_imbd", when((col("nota_media_imbd") >= 0) & (col("nota_media_imbd") <= 10), col("nota_media_imbd")).otherwise(None)) \
    .withColumn("qtd_votos_tmdb", when(col("qtd_votos_tmdb") >= 0, col("qtd_votos_tmdb")).otherwise(None)) \
    .withColumn("qtd_votos_imbd", when(col("qtd_votos_imbd") >= 0, col("qtd_votos_imbd")).otherwise(None))

df_met.selectExpr(
    "id as id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
    "nota_media_imbd", "qtd_votos_imbd"
).write.format("delta").mode("overwrite") \
 .option("overwriteSchema", "true") \
 .saveAsTable("silver.tb_metricas_engajamento")

# 4) silver.tb_avaliacoes_usuarios
df_reviews = spark.read.table("bronze.tb_movies_reviews") \
    .dropDuplicates(["id", "nome", "nota", "comentario"])
df_reviews = df_reviews.withColumn("nota_num", expr("try_cast(nota as decimal(4,1))"))
df_reviews = df_reviews \
    .withColumn("nota", when((col("nota_num") >= 0) & (col("nota_num") <= 10), col("nota_num")).otherwise(None)) \
    .withColumn("comentario", when(col("comentario").isNull() | (trim(col("comentario")) == ""), "Sem comentário").otherwise(col("comentario"))) \
    .drop("nota_num")

df_reviews.selectExpr(
    "id as id_filme", "nome as nome_usuario",
    "nota as nota_usuario", "comentario as comentario_usuario"
).write.format("delta").mode("overwrite") \
 .option("overwriteSchema", "true") \
 .saveAsTable("silver.tb_avaliacoes_usuarios")

# 5) silver.tb_generos
df_credits = spark.read.table("bronze.tb_credits_and_tags")
df_genres = df_credits \
    .withColumn("genres_clean", regexp_replace(col("genres"), ";", ",")) \
    .withColumn("genre_split", explode(split(col("genres_clean"), ","))) \
    .selectExpr("id as id_filme", "trim(genre_split) as genero") \
    .filter("genero != ''") \
    .filter("genero RLIKE '^[A-Za-zÀ-ÿ ]+$'") \
    .distinct()

df_genres.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_generos")

# 6) silver.tb_pessoas_empresas
cols_entidades = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora"
}

dfs = []
for campo, tipo in cols_entidades.items():
    dfs.append(
        df_credits
        .withColumn("entidade", explode(split(regexp_replace(col(campo), ";", ","), ",")))
        .selectExpr("id as id_filme", "trim(entidade) as nome_entidade", f"'{tipo}' as tipo_entidade")
        .filter("nome_entidade != ''")
        .filter(r"nome_entidade RLIKE '^[A-Za-zÀ-ÿ0-9 .&\\-]+$'")
    )

df_pessoas_empresas = dfs[0]
for d in dfs[1:]:
    df_pessoas_empresas = df_pessoas_empresas.unionByName(d)

df_pessoas_empresas = df_pessoas_empresas \
    .withColumn("nome_entidade", initcap(col("nome_entidade"))) \
    .distinct()

df_pessoas_empresas.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.tb_pessoas_empresas")